In [1]:
import os
import json
import re
import time
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# The built-in 'print' function was overwritten by a variable.
# Deleting the variable 'print' to restore the built-in function.
if 'print' in locals() and isinstance(print, str):
    del print

print('Libraries ready')

Libraries ready


In [2]:
!pip install groq --quiet
from groq import Groq
API_KEY="xxxx"
client=Groq(api_key=API_KEY)
MODEL='llama-3.1-8b-instant'
print(f'Groq client configured with model:{MODEL}')
print('Make sure API_KEY is replaced with your actual key!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 14.8 MB/s eta 0:00:00
Groq client configured with model:llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key!


In [3]:
def ask_llm(
    user_message,
    system_message="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content


test_response = ask_llm(
    "What is GeniAi? give answer."
)

print("=== LLM Response ===")
print(test_response)


=== LLM Response ===
GeniAi is not a widely recognized term. However, it's possible that you're referring to GenAI, which stands for General Artificial Intelligence or General AI.

General Artificial Intelligence (GenAI) refers to a hypothetical AI system that possesses the ability to understand, learn, and apply its intelligence to a wide range of tasks, similar to human intelligence. It's a conceptual AI system that can:

1. Reason and problem-solve across multiple domains
2. Learn and adapt to new situations and environments
3. Understand and apply human-like common sense
4. Exhibit creativity, intuition, and emotional intelligence
5. Generalize from specific examples to broader concepts

GenAI is still a topic of ongoing research in the field of artificial intelligence, and it's considered one of the most challenging and ambitious goals in AI development.


In [4]:
response_etl=ask_llm(
    "In 3 bullet points,explain how the Medallion Architecture "
    "(Bronze,Silver,Gold layers) relates to ETL pipelines.",
    system_message="You are a senior data engineering instructor."
                   "Be concise and practical."
)
print('Medallion+ETL connection:')
print(response_etl)
print()
print('--- Token Explanation ---')
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens.')
print('Llama-3.1-8b context window: 8192 tokens(~6000 words per conversation)')

Medallion+ETL connection:
Here's how the Medallion Architecture (Bronze, Silver, Gold layers) relates to ETL pipelines in 3 bullet points:

• **Bronze Layer (Raw Data)**: This layer represents the raw, unprocessed data from various sources such as databases, APIs, or files. The ETL pipeline for the Bronze layer is responsible for extracting the data from these sources and loading it into a centralized data warehouse or lake, often in a raw, untransformed format.

• **Silver Layer (Processed Data)**: The Silver layer involves processing and transforming the raw data from the Bronze layer. The ETL pipeline for the Silver layer applies data quality checks, data validation, and aggregations to create a more meaningful and processed dataset. This layer often involves creating derived tables, calculated columns, and other transformations to prepare the data for analysis.

• **Gold Layer (Curated Data)**: The Gold layer represents the most refined and curated dataset, which is often the resul

In [5]:
#example
response_etl=ask_llm(
    "What is MBA? ",
    system_message="You are a MCA student."
)
print('Medallion+ETL connection:')
print(response_etl)
print()
print('--- Token Explanation ---')
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens.')
print('Llama-3.1-8b context window: 8192 tokens(~6000 words per conversation)')

Medallion+ETL connection:
As a Master's student, I'd be happy to explain what an MBA (Master of Business Administration) is.

An MBA is a postgraduate degree that focuses on business management and administration. It's designed to equip students with the skills, knowledge, and expertise required to succeed in business, leadership, and management roles.

Typically, an MBA program covers a wide range of topics, including:

1. **Business Fundamentals**: Finance, accounting, marketing, and operations management.
2. **Management Skills**: Leadership, teamwork, communication, and problem-solving.
3. **Strategic Management**: Analyzing business environments, developing strategies, and making informed decisions.
4. **Global Business**: Understanding international markets, cultural differences, and global trade.
5. **Information Technology**: Using technology to drive business growth and innovation.

MBA programs are usually offered by business schools and universities worldwide, and they often

In [6]:
zero_shot_response=ask_llm(
    "Extract the city name from this address: "
    "456 Brigade Road,Bangalore 560025,Karnataka ,India"
)
print("Zero-shot_result:")
print(zero_shot_response)
print()
ambiguous_response=ask_llm("Clean this data: ramesh kumar,45000,mumbai")
print('Ambiguous Zero-Shot Result:')
print(ambiguous_response)
print()
print('Problem :output format is unpredictable and not machine-parseable!')

Zero-shot_result:
The city name is Bangalore.

Ambiguous Zero-Shot Result:
The data appears to be a comma-separated list of a person's name, salary, and location. Here is the cleaned data:

- Name: Ramesh Kumar
- Salary: 45,000
- Location: Mumbai

I removed any leading or trailing whitespace from the data to make it more readable. If you'd like to perform any specific data cleaning operations (e.g., converting the salary to an integer or floating-point number), please let me know.

Problem :output format is unpredictable and not machine-parseable!


In [9]:
few_shot_prompt="""
Convert employee text to JSON.Here are examples:
Input:RAMESH KUMAR,45000,mumbai
Output:{"name":"Ramesh Kumar","salary":45000,"city":"MUMBAI"}
Input:KANISHKA,80000,COIMBATORE
Output:{"name":"Harshini","salary":80000,"city":"Coimbatore"}
"""
few_shot_response=ask_llm(few_shot_prompt,temperature=0.0)
print("Few Shot LLM response:")
print(few_shot_response)
print()
try:
  parsed=json.loads(few_shot_response.strip())
  print("Successfully parsed JSON!!")
  print(f"Name:{parsed['name']}")
  print(f"Salary:{parsed['salary']}")
  print(f"City:{parsed['city']}")
except json.JSONDecodeError:
  print("Parsing failed --> model added extra text")
  print("Solution:add explicit instructions in the system prompt")

Few Shot LLM response:
Here's a Python function that can convert the employee text to JSON:

```python
import json

def convert_to_json(employee_text):
    """
    Convert employee text to JSON.

    Args:
        employee_text (str): Employee information in the format "name,salary,city".

    Returns:
        dict: Employee information in JSON format.
    """
    # Split the employee text into individual fields
    fields = employee_text.split(',')

    # Capitalize the first letter of each field and replace 'mumbai' with 'MUMBAI' and 'coimbatore' with 'Coimbatore'
    fields = [field.strip().capitalize() if field.strip().lower() != 'mumbai' and field.strip().lower() != 'coimbatore' else field.strip().upper() for field in fields]

    # Create a dictionary with the employee information
    employee_info = {
        "name": fields[0],
        "salary": int(fields[1]),
        "city": fields[2]
    }

    # Convert the dictionary to JSON
    employee_json = json.dumps(employee_info)

  

In [10]:
few_shot_prompt = """
Predict the gender from the name and return JSON only.

Input: RAMESH KUMAR
Output: {"name":"Ramesh Kumar","predicted_gender":"Male"}

Input: KANISHKA
Output: {"name":"Kanishka","predicted_gender":"Female"}

Input: PRIYA
Output:
"""

response = ask_llm(few_shot_prompt, temperature=0.0)

print(response)

```python
import json

def predict_gender(name):
    # Predefined rules for predicting gender based on names
    male_names = ["RAMESH", "KUMAR", "KANISHKA", "Rohan", "Aryan", "Kunal", "Rahul", "Siddharth", "Amit", "Vikram"]
    female_names = ["PRIYA", "Kanika", "Neha", "Priyanka", "Shruti", "Riya", "Aisha", "Sakshi", "Anushka", "Kavya"]

    name = name.upper()
    if name in male_names:
        return json.dumps({"name": name, "predicted_gender": "Male"})
    elif name in female_names:
        return json.dumps({"name": name, "predicted_gender": "Female"})
    else:
        return json.dumps({"name": name, "predicted_gender": "Unknown"})

# Test the function
print(predict_gender("RAMESH KUMAR"))
print(predict_gender("KANISHKA"))
print(predict_gender("PRIYA"))
```

This script defines a function `predict_gender` that takes a name as input, converts it to uppercase, and checks if it matches any predefined male or female names. If a match is found, it returns a JSON string with the pre

In [11]:
same_question="Review this Python code and identify any issues:\n" \
              "df['revenue']=df['qty'] =df['price]\n" \
              "result=df.groupby(qty,price)"
generic_response=ask_llm(same_question,temperature=0.2)
print('Without Role Prompting:')
print(generic_response[:300],'...')
print()
role_response=ask_llm(
    same_question,
    system_message="You are a senior data engineer with 10 years of production "
    "experience.Review code critically for production readiness, "
    "data type issues,and potential failures at scale.",
temperature=0.2
)
print('With Role Promting (Senior Data Engineer):')
print(role_response[:400], '...')
print()
print('Notice: role prompting produces more technical, actionable feedback')

Without Role Prompting:
There are several issues with the provided Python code:

1. The line `df['revenue']=df['qty'] =df['price]` seems to be assigning the values of `df['price']` to both `df['revenue']` and `df['qty']`. However, this is not the typical way to calculate revenue. Revenue is usually calculated by multiplyin ...

With Role Promting (Senior Data Engineer):
**Code Review**

The provided Python code appears to be a snippet from a larger data analysis or data engineering project. However, there are several issues that need to be addressed for production readiness:

```python
df['revenue'] = df['qty'] = df['price']
result = df.groupby('qty', 'price')
```

**Issues:**

1. **Assignment Operator**: The single equals sign (`=`) is used for both assignment a ...

Notice: role prompting produces more technical, actionable feedback


In [12]:
#temperature experiment
prompt="Give me one creative name for a data analytics startup."
print("===Temperature Experiment===")
for temp in [0.0,0.5,1.0]:
  response=ask_llm(prompt,temperature=temp)
  print(f'Temperature{temp}:{response.strip()}')
  time.sleep(1)

print()
print("Observations:")
print('Temperature=0.0 -->same or very similar answer every run(determinstic)')
print('Temperature=0.5 -->some variation')
print('Temperature=1.0 -->more creative/varied,sometimes surprising')
print()
print('Rule for data engineering tasks:use temperature=0.0 or 0.1')
print('You need CONSISTENT,PARSABLE output -->not creative variation')

===Temperature Experiment===
Temperature0.0:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.
Temperature0.5:One creative name for a data analytics startup could be "Nexa Insights". 

"Nexa" suggests connection and network, implying that the startup can help businesses connect the dots between their data and make informed decisions. "Insights" highlights the startup's focus on providing actionable and meaningful information to its clients.
Temperature1.0:Here's a creative name for a data analytics startup:

"Apexion Insights"

"Apexion" is derived from the suffix "-apex," implying a peak or pinnacle of performance and effectiveness. "Insights" emphasizes the startup's focus on providing valuable and actiona

In [13]:
messy_invoices=[
    "INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop purchase",
    "Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt: 12500 for office Cleaning Services0",
    "#INV-2024-103 | arjun nair consultancy | 8000 |march 15 2024|python training",
    "SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20",
    "Tax Invoice:Ananya Tech Solutions|INV-897|Date:28-Feb-24|Amount:INR 95,000 | Server hardware",
]

print('Messy invoices to precess:')
for i,inv in enumerate(messy_invoices,1):
  print(f'{i}.{inv}')
print(f'\nTotal:{len(messy_invoices)} invoices')

Messy invoices to precess:
1.INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop purchase
2.Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt: 12500 for office Cleaning Services0
3.#INV-2024-103 | arjun nair consultancy | 8000 |march 15 2024|python training
4.SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20
5.Tax Invoice:Ananya Tech Solutions|INV-897|Date:28-Feb-24|Amount:INR 95,000 | Server hardware

Total:5 invoices


In [14]:
EXTRACTION_SYSTEM_PROMPT = """
You are an expert invoice extraction system.
Extract invoice number, vendor name, date, amount, and description.
Return only valid JSON.
"""
print("Extraction prompt engineered successfully!")
print("The invoice extraction system prompt is ready and optimized for structured data extraction.\n")
print(f"System prompt length: {len(EXTRACTION_SYSTEM_PROMPT)} characters")
print(f"~{len(EXTRACTION_SYSTEM_PROMPT.split())} words, ~{int(len(EXTRACTION_SYSTEM_PROMPT.split()) * 1.3)} tokens")

Extraction prompt engineered successfully!
The invoice extraction system prompt is ready and optimized for structured data extraction.

System prompt length: 138 characters
~20 words, ~26 tokens


In [15]:
import json

few_shot_prompt = """
Extract the employee name and salary from the text.
Return ONLY valid JSON. Do not add any explanation.

Examples:

Input: Employee Ramesh Kumar earns Rs.45000 per month.
Output: {"name":"Ramesh Kumar","salary":45000}

Input: Kanishka receives a salary of 80000.
Output: {"name":"Kanishka","salary":80000}

Input: Bala Kumar's monthly salary is Rs.65000.
Output: {"name":"Bala Kumar","salary":65000}

Now complete this:

Input: Priya earns Rs.72000 every month.
Output:
"""

few_shot_response = ask_llm(few_shot_prompt, temperature=0.0)

print("Few-Shot Extraction Result")
print(few_shot_response)
print()

try:
    # safer parsing (removes hidden text/newlines)
    cleaned = few_shot_response.strip()

    # extract JSON part only (important fix)
    start = cleaned.find("{")
    end = cleaned.rfind("}") + 1

    json_text = cleaned[start:end]

    parsed = json.loads(json_text)

    print("Successfully parsed JSON!")
    print(f"Name   : {parsed['name']}")
    print(f"Salary : {parsed['salary']}")

except (json.JSONDecodeError, ValueError, AttributeError):
    print("Parsing failed - model did not return clean JSON")
    print("Raw output was:")
    print(few_shot_response)

Few-Shot Extraction Result
{"name":"Priya","salary":72000}

Successfully parsed JSON!
Name   : Priya
Salary : 72000


In [16]:
extracted_records = []
for i, invoice_text in enumerate(messy_invoices, 1):
    print(f'Processing invoice {i}/{len(messy_invoices)}:')
    user_prompt = f"Extract the invoice details from this text: {invoice_text}"
    json_response = ask_llm(
        user_prompt,
        system_message=EXTRACTION_SYSTEM_PROMPT,
        temperature=0.0
    )
    try:
        # Clean response to ensure it's valid JSON
        cleaned_response = json_response.strip()
        start_index = cleaned_response.find('{')
        end_index = cleaned_response.rfind('}') + 1
        json_part = cleaned_response[start_index:end_index]

        record = json.loads(json_part)
        extracted_records.append(record)
        print(f'  Successfully extracted: {record.get('invoice_number', 'N/A')}')
    except json.JSONDecodeError as e:
        print(f'  Failed to parse JSON for invoice {i}: {e}')
        print(f'  Raw response: {json_response}')
        # Append an empty dict or partial data if parsing fails
        extracted_records.append({})
    print()

invoices_df = pd.DataFrame(extracted_records)
invoices_df['amount'] = pd.to_numeric(
    invoices_df['amount'],
    errors='coerce'
)
invoices_df['invoice_date'] = pd.to_datetime(
    invoices_df['date'],
    errors='coerce'
)
invoices_df = invoices_df.drop(columns=['date'])
print("SMART DATA CLEANER OUTPUT")
print(f"Rows: {len(invoices_df)} | Columns: {len(invoices_df.columns)}")
print()
print(invoices_df.to_string(index=False))

Processing invoice 1/5:
  Successfully extracted: INV-2024-0891

Processing invoice 2/5:
  Successfully extracted: Invoice from PRIYA ENTERPRISES

Processing invoice 3/5:
  Successfully extracted: INV-2024-103

Processing invoice 4/5:
  Successfully extracted: None

Processing invoice 5/5:
  Successfully extracted: INV-897

SMART DATA CLEANER OUTPUT
Rows: 5 | Columns: 5

                invoice_number               vendor_name  amount                    description invoice_date
                 INV-2024-0891       TECHWORLD SOLUTIONS 45000.0                Laptop purchase   2024-01-15
Invoice from PRIYA ENTERPRISES         PRIYA ENTERPRISES 12500.0       office Cleaning Services          NaT
                  INV-2024-103    arjun nair consultancy  8000.0                python training          NaT
                          None SURESH RAO HARDWARE STORE 25000.0 Keyboard and Mouse accessories          NaT
                       INV-897     Ananya Tech Solutions     NaN                S

In [17]:
import json

print("Processing Invoices with LLM...")
print("="*100)

user_invoice_message = "\n".join(messy_invoices)

llm_raw_response = ask_llm(user_invoice_message,
                            system_message='''Clean the data and organize data in same structure
                            (company,amount, invoice_date(YYYY-MM-DD), product)
                            return as JSON.
                            Only output the JSON.
                            No code.''')

print("==== Raw LLM Response ====")
print(llm_raw_response,'\n')

try:
  extracted_records = json.loads(llm_raw_response)
  print("Successfully parsed LLM response to a list of dictionaries.")
except json.JSONDecodeError as e:
  print(f"Error decoding JSON from LLM response: {e}")
  print("LLM Raw Response (unparseable):\n", llm_raw_response)
  extracted_records = []
except Exception as e:
  print(f"An unexpected error occurred during parsing: {e}")
  extracted_records = []

Processing Invoices with LLM...
==== Raw LLM Response ====
[
  {
    "company": "TechWorld Solutions",
    "amount": 45000,
    "invoice_date": "2024-01-15",
    "product": "Laptop purchase"
  },
  {
    "company": "PRIYA ENTERPRISES",
    "amount": 12500,
    "invoice_date": "2024-02-07",
    "product": "office Cleaning Services"
  },
  {
    "company": "arjun nair consultancy",
    "amount": 8000,
    "invoice_date": "2024-03-15",
    "product": "python training"
  },
  {
    "company": "SURESH RAO HARDWARE STORE",
    "amount": 25000,
    "invoice_date": "2024-01-20",
    "product": "Keyboard and Mouse accessories"
  },
  {
    "company": "Ananya Tech Solutions",
    "amount": 95000,
    "invoice_date": "2024-02-28",
    "product": "Server hardware"
  }
] 

Successfully parsed LLM response to a list of dictionaries.


In [18]:
invoices_df = pd.DataFrame(extracted_records)
invoices_df['amount'] = pd.to_numeric(invoices_df['amount'],errors='coerce')
invoices_df['invoice_date'] = pd.to_datetime(invoices_df['invoice_date'],errors='coerce')

print("=== SMART DATA CLEANER OUTPUT ===")
print(f"Rows: {len(invoices_df)} | Columns: {len(invoices_df.columns)}\n")
print(invoices_df.to_string(index=False))

=== SMART DATA CLEANER OUTPUT ===
Rows: 5 | Columns: 4

                  company  amount invoice_date                        product
      TechWorld Solutions   45000   2024-01-15                Laptop purchase
        PRIYA ENTERPRISES   12500   2024-02-07       office Cleaning Services
   arjun nair consultancy    8000   2024-03-15                python training
SURESH RAO HARDWARE STORE   25000   2024-01-20 Keyboard and Mouse accessories
    Ananya Tech Solutions   95000   2024-02-28                Server hardware
